In [3]:
import numpy as np

# -------------------------
# Data
# -------------------------
X = np.array([
    [1.0, 10.0],
    [2.0, 15.0],
    [3.0, 20.0],
    [4.0, 25.0],
], dtype=float)

y = np.array([18.0, 35.0, 65.0, 110.0], dtype=float)


# -------------------------
# Model and loss
# -------------------------
def f_theta(theta, x):
    theta1, theta2, theta3 = theta
    x1, x2 = x
    return theta1 * np.tanh(theta2 * x1) + theta3 * (x2 ** 2)


def loss(theta, X, y):
    preds = np.array([f_theta(theta, x) for x in X])
    return np.mean((y - preds) ** 2)


def grad_single(theta, x, y_i):
    theta1, theta2, theta3 = theta
    x1, x2 = x

    z = theta2 * x1
    tanh_z = np.tanh(z)
    sech2_z = 1.0 - tanh_z**2

    f = theta1 * tanh_z + theta3 * (x2 ** 2)
    err = y_i - f

    df_dtheta1 = tanh_z
    df_dtheta2 = theta1 * x1 * sech2_z
    df_dtheta3 = x2 ** 2

    g1 = -2.0 * err * df_dtheta1
    g2 = -2.0 * err * df_dtheta2
    g3 = -2.0 * err * df_dtheta3

    g = np.array([g1, g2, g3], dtype=float)

    # gradient clipping to stop explosion
    g = np.clip(g, -1000.0, 1000.0)
    return g


# -------------------------
# SGD
# -------------------------
def sgd_train(
    X,
    y,
    theta0,
    lr=1e-6,
    epochs=20000,
    seed=0,
    decay=0.9999,
):
    rng = np.random.default_rng(seed)
    theta = theta0.astype(float).copy()
    history = []

    n = len(X)
    step_size = lr

    for epoch in range(epochs):
        idx = rng.permutation(n)

        for i in idx:
            g = grad_single(theta, X[i], y[i])

            # stop this run if unstable
            if not np.all(np.isfinite(g)) or not np.all(np.isfinite(theta)):
                return None, history

            theta -= step_size * g

            if not np.all(np.isfinite(theta)):
                return None, history

        step_size *= decay

        if epoch % 500 == 0 or epoch == epochs - 1:
            cur_loss = loss(theta, X, y)
            if not np.isfinite(cur_loss):
                return None, history
            history.append((epoch, theta.copy(), cur_loss))

    return theta, history


# -------------------------
# Multiple random restarts
# -------------------------
def fit_with_restarts(
    X,
    y,
    n_restarts=30,
    lr=1e-6,
    epochs=20000,
    seed=123,
):
    rng = np.random.default_rng(seed)

    best_theta = None
    best_loss = float("inf")
    best_history = None

    for r in range(n_restarts):
        # safer initialisation
        theta0 = np.array([
            rng.uniform(0.0, 20.0),     # theta1
            rng.uniform(-0.5, 0.5),     # theta2
            rng.uniform(0.0, 0.3),      # theta3
        ])

        theta, hist = sgd_train(
            X, y,
            theta0=theta0,
            lr=lr,
            epochs=epochs,
            seed=seed + r,
            decay=0.9999,
        )

        if theta is None:
            continue

        cur_loss = loss(theta, X, y)

        if np.isfinite(cur_loss) and cur_loss < best_loss:
            best_loss = cur_loss
            best_theta = theta.copy()
            best_history = hist

    return best_theta, best_loss, best_history


# -------------------------
# Run Qn 3
# -------------------------
best_theta, best_loss, history = fit_with_restarts(
    X, y,
    n_restarts=30,
    lr=1e-4,
    epochs=20000,
    seed=123,
)

print("========== QN 3 ==========")

if best_theta is None:
    print("Training failed for all restarts. Try an even smaller learning rate.")
else:
    print("Estimated theta* =", best_theta)
    print("Final loss L(theta*) =", best_loss)

    print("\nPredictions:")
    for i, x in enumerate(X):
        pred = f_theta(best_theta, x)
        print(f"x{i+1} = {x}, y{i+1} = {y[i]:.6f}, pred = {pred:.6f}")

    print("\nTraining snapshots:")
    for epoch, theta_snap, loss_snap in history[:5]:
        print(f"epoch={epoch:6d} | theta={theta_snap} | loss={loss_snap:.8f}")
    print("...")
    for epoch, theta_snap, loss_snap in history[-5:]:
        print(f"epoch={epoch:6d} | theta={theta_snap} | loss={loss_snap:.8f}")

========== QN 3 ==========
Estimated theta* = [76.63560729 -0.31374793  0.32482529]
Final loss L(theta*) = 237.39246449265733

Predictions:
x1 = [ 1. 10.], y1 = 18.000000, pred = 9.197346
x2 = [ 2. 15.], y2 = 35.000000, pred = 30.451347
x3 = [ 3. 20.], y3 = 65.000000, pred = 73.542159
x4 = [ 4. 25.], y4 = 110.000000, pred = 137.900258

Training snapshots:
epoch=     0 | theta=[16.4781629  -0.2570914   0.22244012] | loss=123.68199687
epoch=   500 | theta=[18.18525384 -0.85414347  0.12279069] | loss=1179.89354244
epoch=  1000 | theta=[20.82279828 -0.67510492  0.28545462] | loss=812.98711929
epoch=  1500 | theta=[23.30515604 -0.48913165  0.3212134 ] | loss=1721.30486640
epoch=  2000 | theta=[25.58739056 -0.58298787  0.32221497] | loss=1561.86718118
...
epoch= 18000 | theta=[75.07732174 -0.39551567  0.36157743] | loss=655.26848543
epoch= 18500 | theta=[75.57188889 -0.32925106  0.32854977] | loss=271.40416000
epoch= 19000 | theta=[75.9266694  -0.29828015  0.30915239] | loss=135.54823749
epo